# Timepill — AIHub YOLO 학습 노트북

**순서:** 셀 1 → 2 → 3 → 4 → 5 → 6 → 7

런타임이 끊겨도 각 셀은 독립적으로 재실행 가능.

In [ ]:
# ── 셀 1: Drive 마운트 + 경로 설정 ─────────────────────────────
from google.colab import drive
# force_remount=True: Errno 107 (FUSE 끊김) 발생 시 확실하게 재연결
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

DRIVE       = Path('/content/drive/MyDrive')
BASE        = DRIVE / 'aihub이미지데이터/01.데이터/1.Training'
TS_ZIP      = BASE / '원천데이터/단일경구약제 5000종/TS_38_단일.zip'
TL_ZIP      = BASE / '라벨링데이터/단일경구약제 5000종/TL_38_단일.zip'

IMAGES_DIR  = Path('/content/aihub_images')
LABELS_DIR  = Path('/content/aihub_labels')
DATASET     = Path('/content/dataset')
SCRIPT      = DRIVE / 'ml/preprocess_aihub.py'

PILL_YOLOV8 = DRIVE / 'pill.yolov8'
HARD_NEG    = DRIVE / 'hard_negatives'
RUN_NAME    = 'pill_aihub_v1'

print('Drive 마운트 완료')
print('BASE 존재:', BASE.exists())
print('TS_ZIP 존재:', TS_ZIP.exists())
print('TL_ZIP 존재:', TL_ZIP.exists())
print('스크립트 존재:', SCRIPT.exists())

In [ ]:
# ── 셀 2: ZIP 압축 해제 ───────────────────────────────────────────
# Drive FUSE를 통해 직접 추출 — shutil.copy 없음 (Errno 107 방지)
import zipfile
from pathlib import Path

def extract_if_needed(zip_src, dst_dir, marker_name='.extracted'):
    marker = dst_dir / marker_name
    if marker.exists():
        n = sum(1 for _ in dst_dir.rglob('*') if _.is_file())
        print(f'  {dst_dir.name}: 이미 해제됨 ({n}개), 스킵')
        return
    print(f'  {zip_src.name} 해제 중 (Drive 직접 읽기)...')
    dst_dir.mkdir(parents=True, exist_ok=True)
    # Drive에서 직접 열어 청크 단위로 추출 — 대용량 RAM 스파이크 없음
    with zipfile.ZipFile(str(zip_src), 'r') as z:
        members = z.namelist()
        for i, member in enumerate(members):
            z.extract(member, dst_dir)
            if i % 500 == 0:
                print(f'    {i}/{len(members)}...')
    marker.touch()
    print(f'  완료')

print('압축 해제 시작...')
extract_if_needed(TS_ZIP, IMAGES_DIR)
extract_if_needed(TL_ZIP, LABELS_DIR)
print('완료')

In [ ]:
# ── 셀 3: 전처리 (COCO→YOLO, drug_N 단위 split, RAM 최적화) ───────
import subprocess, sys
from pathlib import Path

marker = DATASET / '.preprocessed'
if marker.exists():
    print('전처리 이미 완료됨, 스킵')
else:
    result = subprocess.run([sys.executable, str(SCRIPT)], capture_output=False)
    if result.returncode == 0:
        marker.touch()
        print('전처리 완료')
    else:
        print('전처리 실패 — 위 오류 확인')

In [ ]:
# ── 셀 4: 기존 데이터 병합 (pill.yolov8 + hard_negatives) ─────────
import os, shutil, random
from pathlib import Path

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def list_images(folder):
    if not Path(folder).exists(): return []
    return sorted(p for p in Path(folder).rglob('*')
                  if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def symlink_safe(src, dst):
    if not Path(dst).exists():
        os.symlink(Path(src).resolve(), dst)

added = {'train': 0, 'val': 0}

if PILL_YOLOV8.exists():
    pv8_imgs = [
        img for img in list_images(PILL_YOLOV8 / 'train' / 'images')
        if (PILL_YOLOV8 / 'train' / 'labels' / (img.stem + '.txt')).exists()
    ]
    random.shuffle(pv8_imgs)
    n_val = max(1, int(len(pv8_imgs) * 0.25))
    for dst_split, imgs in [('val', pv8_imgs[:n_val]), ('train', pv8_imgs[n_val:])]:
        for img in imgs:
            lbl = PILL_YOLOV8 / 'train' / 'labels' / (img.stem + '.txt')
            symlink_safe(img, DATASET / 'images' / dst_split / ('pv8_' + img.name))
            dst_lbl = DATASET / 'labels' / dst_split / ('pv8_' + img.stem + '.txt')
            if not dst_lbl.exists(): shutil.copy2(lbl, dst_lbl)
            added[dst_split] += 1
    print(f'pill.yolov8: train +{added["train"]}, val +{added["val"]}')
else:
    print('pill.yolov8 없음, 스킵')

if HARD_NEG.exists():
    negs = list_images(HARD_NEG)
    random.shuffle(negs)
    n_val = max(1, int(len(negs) * 0.2))
    for split, imgs in [('val', negs[:n_val]), ('train', negs[n_val:])]:
        for i, img in enumerate(imgs):
            symlink_safe(img, DATASET / 'images' / split / f'neg_{i:04d}_{img.name}')
            (DATASET / 'labels' / split / f'neg_{i:04d}_{img.stem}.txt').touch()
    print(f'hard_negatives: {len(negs)}장')
else:
    print('hard_negatives 없음, 스킵')

for split in ['train', 'val']:
    imgs = list((DATASET/'images'/split).iterdir()) if (DATASET/'images'/split).exists() else []
    lbls = list((DATASET/'labels'/split).iterdir()) if (DATASET/'labels'/split).exists() else []
    bbox = sum(1 for l in lbls if l.stat().st_size > 0)
    print(f'{split}: {len(imgs)}장 (bbox {bbox} / negative {len(lbls)-bbox})')

In [ ]:
# ── 셀 4b: Synthetic Positives 생성 ──────────────────────────────
# 알약 컷아웃(배경제거 PNG) × 배경 이미지 → YOLO labeled 합성 이미지
# 목표: ~1,200장, train에만 추가

from PIL import Image, ImageEnhance
import random, os
from pathlib import Path

SAMPLE_IMG_DIR  = DRIVE / 'sample_img'    # 알약 컷아웃 PNG (배경제거)
BACKGROUNDS_DIR = DRIVE / 'backgrounds'   # 배경 이미지
SYNTH_TARGET    = 1200
SYNTH_PREFIX    = 'syn_'

IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg'}

def _list_imgs(folder):
    if not Path(folder).exists():
        return []
    return sorted(p for p in Path(folder).rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)

def augment_background(bg: Image.Image) -> Image.Image:
    """배경 이미지 augmentation — 같은 배경을 다양하게 보이게."""
    bg_w, bg_h = bg.size

    # 1. Random crop (zoom in 0~25%) — 다른 시점처럼 보이게
    zoom = random.uniform(0.0, 0.25)
    if zoom > 0.01:
        left   = int(random.uniform(0, bg_w * zoom))
        top    = int(random.uniform(0, bg_h * zoom))
        right  = bg_w - int(bg_w * zoom - left)
        bottom = bg_h - int(bg_h * zoom - top)
        right  = max(left + 1, min(right, bg_w))
        bottom = max(top  + 1, min(bottom, bg_h))
        bg = bg.crop((left, top, right, bottom)).resize((bg_w, bg_h), Image.LANCZOS)

    # 2. Horizontal flip (50%)
    if random.random() < 0.5:
        bg = bg.transpose(Image.FLIP_LEFT_RIGHT)

    # 3. Color jitter — 색조/채도
    if random.random() < 0.5:
        bg = ImageEnhance.Color(bg).enhance(random.uniform(0.6, 1.4))

    # 4. 밝기
    if random.random() < 0.4:
        bg = ImageEnhance.Brightness(bg).enhance(random.uniform(0.6, 1.4))

    return bg

pills = _list_imgs(SAMPLE_IMG_DIR)
bgs   = _list_imgs(BACKGROUNDS_DIR)
print(f'알약 컷아웃: {len(pills)}장, 배경: {len(bgs)}장')

if not pills or not bgs:
    print('SAMPLE_IMG_DIR 또는 BACKGROUNDS_DIR 없음 — 스킵')
else:
    out_imgs = DATASET / 'images' / 'train'
    out_lbls = DATASET / 'labels' / 'train'
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    existing = len(list(out_imgs.glob(f'{SYNTH_PREFIX}*.jpg')))
    if existing >= SYNTH_TARGET:
        print(f'합성 이미지 이미 {existing}장 존재 — 스킵')
    else:
        need = SYNTH_TARGET - existing
        print(f'{need}장 생성 시작...')

        generated = 0
        attempts  = 0

        while generated < need and attempts < SYNTH_TARGET * 10:
            attempts += 1
            try:
                pill_path = random.choice(pills)
                bg_path   = random.choice(bgs)

                bg   = Image.open(bg_path).convert('RGB')
                bg_w, bg_h = bg.size

                # 배경 augmentation (배경 수가 적어도 다양성 확보)
                bg = augment_background(bg)
                bg_w, bg_h = bg.size

                pill = Image.open(pill_path).convert('RGBA')

                # 알약 크기: 배경 짧은 변의 10~35%
                scale    = random.uniform(0.10, 0.35)
                pill_max = int(min(bg_w, bg_h) * scale)
                pw, ph   = pill.size
                ratio    = min(pill_max / max(pw, 1), pill_max / max(ph, 1))
                pill     = pill.resize((max(1, int(pw * ratio)), max(1, int(ph * ratio))), Image.LANCZOS)

                # 랜덤 회전 ±30도
                pill = pill.rotate(random.uniform(-30, 30), expand=True, resample=Image.BICUBIC)
                pw, ph = pill.size

                if bg_w <= pw or bg_h <= ph:
                    continue

                # 배경 위 랜덤 위치에 붙여넣기
                x = random.randint(0, bg_w - pw)
                y = random.randint(0, bg_h - ph)
                composite = bg.copy()
                composite.paste(pill, (x, y), mask=pill.split()[3])

                # 합성 이미지 대비 조정 (알약-배경 경계 자연스럽게)
                if random.random() < 0.4:
                    composite = ImageEnhance.Contrast(composite).enhance(random.uniform(0.8, 1.2))

                # 640×640으로 리사이즈
                OUT = 640
                composite = composite.resize((OUT, OUT), Image.LANCZOS)

                # YOLO bbox 재계산
                sx, sy = OUT / bg_w, OUT / bg_h
                cx  = (x * sx + pw * sx / 2) / OUT
                cy  = (y * sy + ph * sy / 2) / OUT
                bwn = (pw * sx) / OUT
                bhn = (ph * sy) / OUT
                cx, cy   = max(0.0, min(1.0, cx)),  max(0.0, min(1.0, cy))
                bwn, bhn = max(0.0, min(1.0, bwn)), max(0.0, min(1.0, bhn))

                # bbox가 너무 작으면 스킵
                if bwn < 0.05 or bhn < 0.05:
                    continue

                fname = f'{SYNTH_PREFIX}{existing + generated:05d}'
                composite.save(out_imgs / f'{fname}.jpg', quality=92)
                with open(out_lbls / f'{fname}.txt', 'w') as f:
                    f.write(f'0 {cx:.6f} {cy:.6f} {bwn:.6f} {bhn:.6f}\n')

                generated += 1
                if generated % 300 == 0:
                    print(f'  {generated}/{need}장...')

            except Exception:
                pass

        print(f'합성 완료: {generated}장 → train 추가')


In [ ]:
# ── 셀 5: data.yaml ──────────────────────────────────────────────
from pathlib import Path
yaml_path = Path('/content/data.yaml')
yaml_path.write_text(
    f'path: {DATASET}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ["pill"]\n'
)
print(yaml_path.read_text())

In [ ]:
# ── 셀 6: YOLO 학습 ──────────────────────────────────────────────
!pip install ultralytics -q

from ultralytics import YOLO
from pathlib import Path
import shutil

def make_drive_callback(drive_path, run_name, every=10):
    def _cb(trainer):
        if (trainer.epoch + 1) % every == 0:
            src = Path(f'/content/runs/{run_name}/weights/last.pt')
            if src.exists():
                dst = drive_path / f'ml/{run_name}_epoch{trainer.epoch + 1}.pt'
                shutil.copy2(src, dst)
                print(f'  Drive 저장 완료: {dst.name}')
    return _cb

model = YOLO('yolo11n.pt')
model.add_callback('on_train_epoch_end', make_drive_callback(DRIVE, RUN_NAME, every=10))
model.train(
    data='/content/data.yaml',
    epochs=25, imgsz=640, batch=16, patience=20,
    degrees=0.0, translate=0.03, scale=0.10,
    shear=0.0, perspective=0.0002,
    fliplr=0.0, flipud=0.0,
    mosaic=0.8, hsv_h=0.02, hsv_s=0.20, hsv_v=0.18,
    project='/content/runs', name=RUN_NAME, exist_ok=True,
    save=True, save_period=10,
)
BEST_PT = Path(f'/content/runs/{RUN_NAME}/weights/best.pt')
print('학습 완료:', BEST_PT)

In [ ]:
# ── 셀 7: 평가 + TFLite export + Drive 저장 ──────────────────────
import shutil
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image as IPImage, display

RUN_DIR = Path(f'/content/runs/{RUN_NAME}')
BEST_PT = RUN_DIR / 'weights/best.pt'
assert BEST_PT.exists(), f'best.pt 없음: {BEST_PT}'

for name in ['results.png', 'PR_curve.png', 'confusion_matrix.png']:
    p = RUN_DIR / name
    if p.exists(): display(IPImage(str(p)))

best = YOLO(str(BEST_PT))
m = best.val(data='/content/data.yaml', split='val', imgsz=640)
print(f'mAP50={m.box.map50:.4f}  mAP50-95={m.box.map:.4f}  P={m.box.mp:.4f}  R={m.box.mr:.4f}')

exported = Path(best.export(format='tflite', int8=True,
                            data='/content/data.yaml', imgsz=320))
print('TFLite:', exported)

out_dir = DRIVE / f'{RUN_NAME}_results'
out_dir.mkdir(parents=True, exist_ok=True)
for src in [BEST_PT, exported,
            RUN_DIR/'results.png', RUN_DIR/'PR_curve.png', RUN_DIR/'confusion_matrix.png']:
    if Path(src).exists():
        shutil.copy2(src, out_dir / Path(src).name)
        print('저장:', out_dir / Path(src).name)
print('완료 →', out_dir)